# Feature scaling

## Loading the data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

housing = pd.read_csv("../data/housing.csv")
train_set, test_set = train_test_split(
    housing, test_size=0.2, random_state=42,
    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]))

X_train = train_set.drop("median_house_value", axis=1)  # features
y_train = train_set["median_house_value"].copy()        # target (labels)

X_train_num = X_train.select_dtypes(include=[np.number])  # numerical columns only

## Scaling, normalization and standardization

Most Machine Learning algorithms don't work well when features have very different scales. For example, many algorithms calculate the distance between two points using Euclidean distance. If one of the features has much larger values than the others, the distance will be dominated by this feature. For example, in the California dataset, the range of `median_income` goes from 0 to 15, while the range of `total_rooms` goes from 6 to 39,320.

To avoid this, it's common to scale the features.

The terminology can be confusing at this point. In general, **normalization** refers to changing the scale of data to fit a specific range, while **standardization** refers to changing the distribution of data to have a mean of 0 and a standard deviation of 1. In both cases, they are linear transformations that don't change the shape of the data distribution. In statistics there's usually a clear distinction between both terms, but in deep learning and computer vision, the terminology can be less consistent and **it's common to use "normalization" to refer to standardization**.

In [ ]:
X_train_num.describe().T

,count,mean,std,min,25%,50%,75%,max
longitude,16512.0,-119.575635,2.001828,-124.3500,-121.80000,-118.51000,-118.010000,-114.3100
latitude,16512.0,35.639314,2.137963,32.5400,33.94000,34.26000,37.720000,41.9500
housing_median_age,16512.0,28.653404,12.574819,1.0000,18.00000,29.00000,37.000000,52.0000
total_rooms,16512.0,2622.539789,2138.417080,6.0000,1443.00000,2119.00000,3141.000000,39320.0000
total_bedrooms,16354.0,534.914639,412.665649,2.0000,295.00000,433.00000,644.000000,6210.0000
population,16512.0,1419.687379,1115.663036,3.0000,784.00000,1164.00000,1719.000000,35682.0000
households,16512.0,497.011810,375.696156,2.0000,279.00000,408.00000,602.000000,5358.0000
median_income,16512.0,3.875884,1.904931,0.4999,2.56695,3.54155,4.745325,15.0001


### Why it matters: a distance example

Take two districts and compute the squared difference of each feature, which is what the Euclidean distance adds up. `total_rooms` alone accounts for almost all of the distance; `median_income`, the most predictive feature, is irrelevant to it.

In [ ]:
district_a = X_train_num.iloc[0]
district_b = X_train_num.iloc[1]

squared_differences = (district_a - district_b) ** 2
(squared_differences / squared_differences.sum()).round(4)  # share of each feature in the squared distance

longitude             0.0000
latitude              0.0000
housing_median_age    0.0002
total_rooms           0.9735
total_bedrooms        0.0016
population            0.0229
households            0.0018
median_income         0.0000
dtype: float64

## `MinMaxScaler`

The most common normalization is **min-max normalization** or **min-max scaling**. **Min-max normalization** is the simplest: values are scaled and shifted so that they end up in the range between a minimum value and a maximum value. Normally it will be between 0 and 1, although they can be others (neural networks usually work better with *inputs* with mean 0, so sometimes the range -1 to 1 is used). Scikit-Learn provides a `MinMaxScaler` class for this.

$$ X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}} $$

In [ ]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
housing_num_min_max_scaled = min_max_scaler.fit_transform(X_train_num)

Min-Max normalization is very sensitive to *outliers*, since a single very large value can completely change the scale of the data. In a situation where all data is between 20 and 30 but a single value of 100 appears, the maximum becomes 100, shifting all other values to a very low range. In general, Min-Max normalization should only be used if we're sure that the *outliers* are not errors. Techniques such as capping or Winsorizing can be applied beforehand to limit the impact of extreme values (see [Outliers and capped values](outliers.ipynb)).

## `StandardScaler`

On the other hand, **Z-score standardization** (***standard score***) is different: first it subtracts the mean (so it becomes 0), and then divides by the **standard deviation** so that the resulting distribution has standard deviation 1. Unlike min-max scaling, standardization doesn't limit values to a specific range, but this also has the advantage of being much less sensitive to outliers. Scikit-Learn provides a `StandardScaler` class for this.

$$ X_{std} = \frac{X - \mu}{\sigma} $$

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
housing_num_std_scaled = std_scaler.fit_transform(X_train_num)

## `RobustScaler`

When outliers are present and cannot be removed, `RobustScaler` is a third option: it subtracts the **median** and divides by the **interquartile range** (IQR, the distance between the 25th and 75th percentiles). Both statistics ignore extreme values, so the bulk of the data ends up on a comparable scale regardless of how far the outliers are.

In [ ]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()
housing_num_robust_scaled = robust_scaler.fit_transform(X_train_num)

Comparing the three on `total_rooms` (a heavy-tailed variable) shows the trade-offs: min-max bounds the range but squeezes most values near the minimum; standardization centers the data but leaves the tail far out; the robust scaler puts the median at 0 and makes the central 50% of the data span exactly one unit (from the 25th to the 75th percentile), whatever the tail does.

In [ ]:
column = list(X_train_num.columns).index("total_rooms")
pd.DataFrame({
    "min-max [-1, 1]": housing_num_min_max_scaled[:, column],
    "standard": housing_num_std_scaled[:, column],
    "robust": housing_num_robust_scaled[:, column],
}).describe().round(2)

,"min-max [-1, 1]",standard,robust
count,16512.00,16512.00,16512.00
mean,-0.87,-0.00,0.30
std,0.11,1.00,1.26
min,-1.00,-1.22,-1.24
25%,-0.93,-0.55,-0.40
50%,-0.89,-0.24,0.00
75%,-0.84,0.24,0.60
max,1.00,17.16,21.91


## Scaling the target

Many ML models work better by standardizing input *features* and it's a common and systematic practice in most cases (except for tree-based models). Scaling the *target* is less common, but can be useful in some cases, particularly for gradient-based models (such as neural networks) or distance-based models (such as KNN or SVM regressions).

For example, we could apply `StandardScaler` again to the labels:

In [ ]:
target_scaler = StandardScaler()
scaled_labels = target_scaler.fit_transform(y_train.to_frame())  # fit_transform expects 2D, so the Series becomes a DataFrame
print(type(y_train))  # a single column: the labels were stored in a Series
scaled_labels

<class 'pandas.Series'>


array([[-1.16601465],
       [ 0.62745084],
       [-1.07439665],
       ...,
       [-0.5756836 ],
       [ 0.44162188],
       [-1.2472608 ]], shape=(16512, 1))

### Inverting the transformation

If we transform the target variable in any way, our model's output will also return transformed predictions. If we want the predictions to be on the original scale, we'll need to invert the transformation. Many of Scikit-Learn's transformers have an `inverse_transform()` method, which makes it easy to calculate the inverse of their transformations.

> **Note:** Not all transformations are invertible. For example:
> - **Numerical scalers** (`StandardScaler`, `MinMaxScaler`, `RobustScaler`): Always invertible—they store the parameters needed to reverse the transformation.
> - **`OneHotEncoder`**: Technically invertible via `inverse_transform()`, but information loss occurs if the original categories aren't known.
> - **`OrdinalEncoder`**: Invertible if categories are preserved.
> - **Information-lossy transformations**: Some transformations like quantile-based discretization or heavy rounding cannot be perfectly inverted.

To give a simplified example, we're going to train a **simple linear regression with the most correlated predictor** (`median_income`) and the labels we just scaled. Then we'll make predictions and undo the transformation.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train[["median_income"]], scaled_labels)  # train the model with the scaled target

some_new_data = X_train[["median_income"]].iloc[:5]  # for simplicity, we simulate new inputs with 5 training rows

scaled_predictions = model.predict(some_new_data)
target_scaler.inverse_transform(scaled_predictions)  # undo the transformation to get predictions on the original scale

array([[135958.75805364],
       [309735.008975  ],
       [165232.3998617 ],
       [138162.41971241],
       [232903.1766333 ]])

This process can be simplified using the `TransformedTargetRegressor` class from Scikit-Learn, which allows training a model with transformed labels and undoing the transformation automatically.

In [ ]:
from sklearn.compose import TransformedTargetRegressor

model = TransformedTargetRegressor(regressor=LinearRegression(),
                                   transformer=StandardScaler())  # target transformer
model.fit(X_train[["median_income"]], y_train)
model.predict(some_new_data)

array([135958.75805364, 309735.008975  , 165232.3998617 , 138162.41971241,
       232903.1766333 ])

Two related notes: scaling is a linear transformation and does not fix a skewed shape, for which a logarithm is applied *before* scaling ([Heavy-tailed distributions](heavy_tailed_distributions.ipynb)); and scalers must be fitted on the training set only, which a [pipeline](pipelines.ipynb) guarantees ([Data leakage](data_leakage.ipynb)).